In [ ]:
# 1/4 - clone + dotnet (fault-tolerant)
# 1/4 - clone + dotnet (fault-tolerant)
print('===== ALL NATIVE KERNELS -> PURE TKV (14+7+2) + LOSS CHECK =====')
# 1/4 - clone main + prototype + Inference, install .NET (fault-tolerant)
import subprocess, os, shutil
print('===== INTEGRATE 14 PURE TKV KERNELS INTO NATIVE BUILD + LOSS CHECK =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
subprocess.run('rm -rf /content/TVinf', shell=True)
subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git /content/TVinf', shell=True, capture_output=True, text=True, timeout=600)
dotnet=shutil.which('dotnet')
if not dotnet:
    print('dotnet not found -> installing .NET 10 SDK...')
    p=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && chmod +x /tmp/dotnet-install.sh && /tmp/dotnet-install.sh --channel 10.0 --install-dir /usr/share/dotnet', shell=True, capture_output=True, text=True, timeout=900)
    print('install rc=',p.returncode)
    dotnet='/usr/share/dotnet/dotnet'
print('dotnet =',dotnet,'exists=',os.path.exists(dotnet))

In [ ]:
# 2/4 - build TOAN BO native kernel tu .tkv THUAN (43 exports) + copy .so
import subprocess, shutil, os, re
root='/content/TVmain/Release/tools/simd_native'
ex='/content/TVcpu/Release/cuda_backend/examples'
bwd=['bias_add_rows_cpu.cpu.c','softmax_causal_fwd_pure.cpu.c','merge_hb_head_cpu.cpu.c',
     'layernorm_bwd_cpu.cpu.c','gelu_erf_bwd_pure.cpu.c','gelu_erf_fwd_pure.cpu.c',
     'bias_grad_cpu.cpu.c','softmax_causal_bwd_pure.cpu.c','embed_bwd_pure.cpu.c',
     'layernorm_fwd_cpu.cpu.c']
parts=[]
for f in bwd:
    s=open(ex+'/'+f).read()
    ls=[l for l in s.split('\n') if not l.startswith('#include') and not l.startswith('#define') and l.strip()!='']
    parts.append('\n'.join(ls))
hdr='#include <immintrin.h>\n#include <math.h>\n#include <stdlib.h>\n#define EXPORT __attribute__((visibility("default")))\n'
open(root+'/tkv_bwd_ops.c','w').write(hdr+'\n\n'.join(parts)+'\n')
shutil.copyfile(ex+'/linear_dw_cpu.cpu.c', root+'/tkv_linear_dw.c')
shutil.copyfile(ex+'/linear_dx_cpu.cpu.c', root+'/tkv_linear_dx.c')
p=root+'/tkv_simd_native.c'; s=open(p).read()
for name,gf in [('F32ListAdd','f32_list_add_cpu.cpu.c'),('F32ListCopy','f32_list_copy_cpu.cpu.c')]:
    s=re.sub(r'EXPORT int '+name+r'\(.*?\n\}\n', open(ex+'/'+gf).read().strip()+'\n', s, count=1, flags=re.S)
open(p,'w').write(s)
shutil.copyfile(ex+'/flash_all_cpu.cpu.c', root+'/tkv_flash_attn.c')
shutil.copyfile(ex+'/attn_cuda_stub_cpu.cpu.c', root+'/tkv_attn_cuda.c')
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
r=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('ALL-TKV native build rc=',r.returncode)
assert r.returncode==0, r.stderr[:3000]
shutil.copyfile(root+'/tkv_simd_native.so','/content/TVinf/demo/tkv_simd_native.so')
print('.so copied to Inference demo')

In [ ]:
# 2/4 - PARITY: cung mot weights cho native all-TKV va HF PyTorch, so CE
import ctypes, numpy as np, torch
from transformers import GPT2Config, GPT2LMHeadModel
P=ctypes.POINTER(ctypes.c_float); I32=ctypes.c_int32; I64=ctypes.c_int64
lib=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
lib.TkvFullBegin.argtypes=[I64]*6; lib.TkvFullBegin.restype=ctypes.c_int
lib.TkvFullSetTensor.argtypes=[I64,P,I64]; lib.TkvFullSetTensor.restype=ctypes.c_int
lib.TkvFullSetIds.argtypes=[P,I64]; lib.TkvFullSetIds.restype=ctypes.c_int
lib.TkvFullSetTargets.argtypes=[P,I64]; lib.TkvFullSetTargets.restype=ctypes.c_int
lib.TkvFullSetGrad.argtypes=[P,I64]; lib.TkvFullSetGrad.restype=ctypes.c_int
lib.TkvFullSetOpt.argtypes=[P]*3+[ctypes.c_float]*5+[I64]*2; lib.TkvFullSetOpt.restype=ctypes.c_int
lib.TkvFullStep124m.argtypes=[I64]; lib.TkvFullStep124m.restype=ctypes.c_float
T=1024; V=50257; NL=12; H=768; NH=12; HD=64
torch.manual_seed(7)
cfg=GPT2Config(n_layer=NL,n_head=NH,n_embd=H,n_positions=1024,vocab_size=V,
               layer_norm_epsilon=1e-6, activation_function='gelu',
               resid_pdrop=0.0,attn_pdrop=0.0,embd_pdrop=0.0)
model=GPT2LMHeadModel(cfg).eval()
sd=model.state_dict()
def blocks():
    out=[sd['transformer.wte.weight'].numpy(), sd['transformer.wpe.weight'].numpy()[:T],
         sd['transformer.ln_f.weight'].numpy(), sd['transformer.ln_f.bias'].numpy()]
    for i in range(NL):
        p='transformer.h.%d.'%i
        out += [sd[p+'ln_1.weight'].numpy(), sd[p+'ln_1.bias'].numpy(),
                sd[p+'attn.c_attn.weight'].numpy().copy(), sd[p+'attn.c_attn.bias'].numpy(),
                sd[p+'attn.c_proj.weight'].numpy().copy(), sd[p+'attn.c_proj.bias'].numpy(),
                sd[p+'ln_2.weight'].numpy(), sd[p+'ln_2.bias'].numpy(),
                sd[p+'mlp.c_fc.weight'].numpy().copy(), sd[p+'mlp.c_fc.bias'].numpy(),
                sd[p+'mlp.c_proj.weight'].numpy().copy(), sd[p+'mlp.c_proj.bias'].numpy()]
    return out
params=blocks()
flat=np.concatenate([x.reshape(-1) for x in params]).astype(np.float32)
print('tensors=',len(params),' params=',flat.size)
ids=np.array([(i*37+11)%V for i in range(T)],dtype=np.int32)
tgt=np.array([(i*37+11)%V for i in range(1,T+1)],dtype=np.int32)
grad=np.zeros(flat.size,np.float32); m=np.zeros(flat.size,np.float32); v=np.zeros(flat.size,np.float32)
print('BEGIN',lib.TkvFullBegin(len(params),NL,H,NH,HD,V))
for i,a in enumerate(params):
    aa=np.ascontiguousarray(a,dtype=np.float32)
    r=lib.TkvFullSetTensor(i,aa.ctypes.data_as(P),aa.size)
    if r!=0: print('SET_FAIL',i,r)
print('IDS',lib.TkvFullSetIds(ids.ctypes.data_as(P),T))
print('TGT',lib.TkvFullSetTargets(tgt.ctypes.data_as(P),T))
print('GRAD',lib.TkvFullSetGrad(grad.ctypes.data_as(P),flat.size))
print('OPT',lib.TkvFullSetOpt(flat.ctypes.data_as(P),m.ctypes.data_as(P),v.ctypes.data_as(P),
                            5e-5,0.9,0.999,1e-8,0.0,1,0))
# forward only (run_mode=1 -> khong update weights)
ln=float(lib.TkvFullStep124m(1))
print('NATIVE forward loss (same weights) = %.6f'%ln)
# HF forward, cung weights, cung ids
xi=torch.from_numpy(ids.astype(np.int64)).unsqueeze(0)
yi=torch.from_numpy(tgt.astype(np.int64)).unsqueeze(0)
with torch.no_grad():
    lh=float(model(input_ids=xi,labels=yi).loss)
print('PYTORCH forward loss (same weights) = %.6f'%lh)
print('ABS DIFF = %.6f'%abs(ln-lh))
print('NATIVE ln(V) = %.6f  (reference for uniform logits)'%np.log(V))

In [ ]:
# 3/4 - summary
print('Same-weights parity: if native != PyTorch, difference is due to numerics/implementation, NOT initialization.')